# FACT 3.0 — kNN Graph + Louvain (ENGLISH) with memory between sampling stages

Same pipeline as FACT 2.5, plus a state saved in `STATE_DIR` so that later sampling stages only code the new properties.

**How to use**
1. Stage 1: `STAGE = 1`, `CSV_PATH` = first sample. Run all cells. The grid search picks `k`, `sim_min`, `resolution` and the state is saved.
2. Stage 2: `STAGE = 2`, `CSV_PATH` = CSV with **only the new participants**. Run all cells again. Stage-1 codes and parameters stay frozen; new properties are assigned to existing codes (kNN vote) or form new codes (residuals).
3. Stage 3, 4, ...: same as stage 2.

`LEGACY = True` reproduces FACT 2.5 exactly (file reading, `u_mode="rows"`).

In [1]:
import os, io, re, json, time, hashlib, warnings, datetime
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import adjusted_rand_score
import networkx as nx
from community.community_louvain import best_partition
from matplotlib.backends.backend_pdf import PdfPages

FACT_VERSION = "3.0"

## 0) LOAD DATA

In [2]:

# Options of data "CSLB_raw", "Sit_Conc_raw"
DATA_NAME = "Sit_Conc_raw_stage1"
STAGE = 1                                     # 1, 2, 3, ...  (stage >= 2 needs the state of the previous stage)
CSV_PATH = f"data/{DATA_NAME}.csv"             # stage 1: first sample; stage >= 2: ONLY the new rows
                                              # e.g. CSV_PATH = "data/Sit_Conc_raw_stage2.csv"
OUTPUT_PREFIX = "data/results/en/en"           # files will be prefixed with this + data name + stage
STATE_DIR = f"data/state/en_{DATA_NAME}"       # memory between stages: written in stage 1, extended later
LEGACY = False                                # True = exact FACT 2.5 (file reading, u_mode="rows")
FIXED_PARAMS = None                           # e.g. {"k": 40, "sim_min": 0.33, "resolution": 1.0}: skip the grid (stage 1)
C_TARGET = 0.60
U_MODE = "rows"                               # "rows" (FACT 2.5) or "incidence" (sum of incidences)
SEED = 42

if LEGACY:
    U_MODE = "rows"
PREFIX = f"{OUTPUT_PREFIX}_{DATA_NAME}_stage{STAGE}"
os.makedirs(os.path.dirname(PREFIX), exist_ok=True)

REQUIRED_COLS = {"concept", "participant", "property"}
# typical traces of a bad decoding: "Ã³", "Â", "â€™", replacement char, C1 control chars
MOJIBAKE_RE = re.compile("Ã[\u0080-¿]|Â[\u0080-¿]|â€||[\u0080-\u009f]")

def read_csv_legacy(csv_path):
    """Exact FACT 2.5 read (latin-1, errors ignored, comma then semicolon)."""
    try:
        df = pd.read_csv(csv_path, encoding='ISO-8859-1', encoding_errors='ignore')
        if not REQUIRED_COLS.issubset(df.columns):
            raise ValueError("Columns not found with comma delimiter")
    except Exception:
        df = pd.read_csv(csv_path, sep=";", engine="python", encoding='ISO-8859-1', encoding_errors='ignore')
    return df

def read_csv_robust(csv_path):
    """UTF-8 first; cp1252 / latin-1 only as fallback. Delimiter detected from the header."""
    raw = open(csv_path, "rb").read()
    for enc in ("utf-8-sig", "cp1252", "latin-1"):
        try:
            text = raw.decode(enc)
            break
        except UnicodeDecodeError:
            continue
    if enc != "utf-8-sig":
        warnings.warn(f"{csv_path} is not valid UTF-8; decoded as {enc}. "
                      "Save the file as UTF-8 to avoid surprises.")
    header = text.lstrip("\ufeff").split("\n", 1)[0]
    sep = None
    for cand in (",", ";", "\t"):
        if REQUIRED_COLS.issubset({c.strip().strip('"') for c in header.split(cand)}):
            sep = cand
            break
    if sep is None:
        raise ValueError(f"CSV must have columns {REQUIRED_COLS}; header found: {header!r}")
    return pd.read_csv(io.StringIO(text), sep=sep, dtype={"participant": str})

def warn_mojibake(df, col="property", n_show=5):
    """Warn if the text looks badly decoded (e.g. 'DecisiÃ³n')."""
    s = df[col].astype(str)
    bad = s[s.str.contains(MOJIBAKE_RE)]
    if len(bad):
        warnings.warn(f"{len(bad)} rows in '{col}' look badly encoded, e.g.: "
                      f"{bad.head(n_show).tolist()}")
    return len(bad)

def prefix_participants(df, stage):
    """Participant IDs are prefixed by stage (s1_5, s2_5) so stages never mix."""
    out = df.copy()
    out["participant_raw"] = out["participant"]
    out["participant"] = f"s{stage}_" + out["participant"].astype(str).str.strip()
    return out

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

# Read: UTF-8 first (latin-1 only as fallback); LEGACY reads exactly like FACT 2.5
df = read_csv_legacy(CSV_PATH) if LEGACY else read_csv_robust(CSV_PATH)
required_cols = {"concept","participant","property"}
assert required_cols.issubset(df.columns), f"CSV must have columns {required_cols}"
n_bad_text = warn_mojibake(df)

# s1_5, s2_5, ...: participants of different stages never mix
df = prefix_participants(df, STAGE)
input_sha256 = file_sha256(CSV_PATH)
print(f"Stage {STAGE}: {len(df)} rows, {df['concept'].nunique()} concepts, "
      f"{df['participant'].nunique()} participants (LEGACY={LEGACY}, u_mode={U_MODE})")

Stage 1: 6545 rows, 72 concepts, 84 participants (LEGACY=False, u_mode=rows)


/tmp/ipykernel_1595/1608076919.py:43: UserWarning: data/Sit_Conc_raw_stage1.csv is not valid UTF-8; decoded as cp1252. Save the file as UTF-8 to avoid surprises.
  warnings.warn(f"{csv_path} is not valid UTF-8; decoded as {enc}. "
/tmp/ipykernel_1595/1608076919.py:60: UserWarning: 6545 rows in 'property' look badly encoded, e.g.: ['In Atlanta', 'In Japan', 'In a park', 'In some, someone’s backyard', 'New York Mets, best baseball team']
  warnings.warn(f"{len(bad)} rows in '{col}' look badly encoded, e.g.: "


## 1) LIGHT NORMALIZATION (ENGLISH)


Keep it light to preserve meaning: lowercase, strip punctuation, remove common stopwords. In stage ≥ 2 only the new rows are normalized; the configuration must match the saved one.

In [3]:

start = time.time()

EN_STOP = set("""
the a an and or but so to of for from in on at as by is are was were be been being
this that those these there here it its itself they them their you your we our us he she her his
with without into onto about over under out up down not no yes very more most less least
can could should would may might will shall do does did doing done have has had having
than then if else when while where who whom whose which what why how
""".split())

def normalize_text_en(s: str):
    s = str(s).lower().strip()
    # remove punctuation and symbols; keep letters/numbers/spaces
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    toks = [t for t in s.split() if t and (t not in EN_STOP) and len(t) > 1]
    return " ".join(toks)

# saved in params.json; stage >= 2 must use the same normalization
NORM_CONFIG = {
    "function": "normalize_text_en",
    "lowercase": True,
    "removed_chars_regex": r"[^a-z0-9\s]",
    "min_token_len": 2,
    "stopwords_sha256": hashlib.sha256(" ".join(sorted(EN_STOP)).encode()).hexdigest(),
}

df["property_norm"] = df["property"].astype(str).map(normalize_text_en)

end = time.time()
standardization_time = end - start
print(f"Time of standardization: {standardization_time}")

Time of standardization: 0.021170377731323242


## 2) EMBEDDINGS (E5 ENGLISH)

For English, use the English E5 model; always prefix with 'passage:'. In stage ≥ 2 only the new rows are embedded (`df` holds only them).

In [4]:

start = time.time()

MODEL_NAME = "intfloat/e5-base-v2"   # same model as FACT_2.5_en

def encode_passages_en(texts, encoder=None):
    enc = encoder if encoder is not None else model
    return enc.encode([f"passage: {t}" for t in texts],
                      normalize_embeddings=True,
                      convert_to_numpy=True)

model = SentenceTransformer(MODEL_NAME)
X = encode_passages_en(df["property_norm"].tolist())

end = time.time()
embedding_time = end - start
print(f"Time of embedding: {embedding_time}")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/67.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

... (3 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Time of embedding: 10.755334377288818


## 3) kNN GRAPH BUILDER

In [5]:

def knn_search(X, k, Y=None, chunk=1024):
    """
    Exact cosine kNN with deterministic ties (higher similarity first, then lower index).
    Y=None: neighbors of every row of X among X (k+1 columns, self included, as in knn_graph).
    Y given: neighbors of every row of Y among X (k columns).
    Returns (dist, idx) like sklearn, dist = 1 - cosine similarity.
    Because ties are broken by index, the first k+1 columns of knn_search(X, k_max)
    are exactly knn_search(X, k): the grid can compute the kNN once.
    """
    Xn = np.asarray(X, dtype=np.float32)
    Xn = Xn / np.maximum(np.linalg.norm(Xn, axis=1, keepdims=True), 1e-12)
    if Y is None:
        Qn, kk = Xn, min(k + 1, len(Xn))
    else:
        Qn = np.asarray(Y, dtype=np.float32)
        Qn = Qn / np.maximum(np.linalg.norm(Qn, axis=1, keepdims=True), 1e-12)
        kk = min(k, len(Xn))
    idx = np.empty((len(Qn), kk), dtype=np.int64)
    sim = np.empty((len(Qn), kk), dtype=np.float32)
    for a in range(0, len(Qn), chunk):
        S = Qn[a:a + chunk] @ Xn.T
        if kk < S.shape[1]:
            kth = -np.partition(-S, kk - 1, axis=1)[:, kk - 1]
        else:
            kth = S.min(axis=1)
        for r in range(S.shape[0]):
            cand = np.flatnonzero(S[r] >= kth[r])              # ascending index
            order = np.argsort(-S[r, cand], kind="stable")[:kk]
            idx[a + r] = cand[order]
            sim[a + r] = S[r, cand[order]]
    return 1.0 - sim.astype(np.float64), idx

def knn_graph(X, k=15, sim_min=0.36, nn=None):
    # nn=(dist, idx) precomputed with k_max >= k; None = original FACT 2.5 behavior
    if nn is None:
        nbrs = NearestNeighbors(n_neighbors=k+1, metric="cosine").fit(X)
        dist, idx = nbrs.kneighbors(X)
    else:
        dist, idx = nn[0][:, :k+1], nn[1][:, :k+1]
    G = nx.Graph(); G.add_nodes_from(range(len(X)))
    add_edge = G.add_edge
    for i in range(len(X)):
        for j, d in zip(idx[i], dist[i]):
            sim = 1.0 - float(d)
            if sim >= sim_min and i != j:
                add_edge(i, int(j), weight=sim)
    return G

## 4) LOUVAIN

In [6]:

def louvain_labels(G, resolution=1.0, seed=42):
    part = best_partition(G, resolution=resolution, random_state=seed)
    labels = np.full(G.number_of_nodes(), -1, dtype=int)
    for n, c in part.items():
        labels[n] = c
    return labels

## 5) POST: SINGLETONS AS UNIQUE CODES

In [7]:
def assign_singletons_as_unique(df_in, labels):
    out = df_in.copy()
    out["code"] = labels
    if (out["code"] == -1).any():
        mask = out["code"] == -1
        max_lbl = out["code"].max() if (out["code"]!=-1).any() else -1
        out.loc[mask, "code"] = np.arange(mask.sum(), dtype=int) + max_lbl + 1
    return out

## 6) COVERAGE & LOSS

In [8]:

def chao_coverage(U, T, Q1, Q2):
    """C = 1 - (Q1/U) * ( Q1*(T-1) / ( Q1*(T-1) + 2*Q2 ) )"""
    if T <= 1:
        return 0.0
    denom = Q1 * (T - 1) + 2 * Q2
    if denom > 0:
        return 1 - (Q1 / U) * (Q1 * (T - 1) / denom)
    # no information to correct: low coverage
    return 1 - (Q1 / U)

def cover_loss(df_codes, C_target=0.60, u_mode="rows"):
    """
    Coverage per concept (Chao, incidence data) and loss = mean (C - C_target)^2.
        U  = "rows":      number of raw properties (rows) of the concept  [FACT 2.5]
             "incidence": sum of incidences = unique (participant, code) pairs
        T  = number of distinct participants of the concept
        Q1 = number of codes mentioned by exactly 1 participant
        Q2 = number of codes mentioned by exactly 2 participants
    Loss and C-bar use u_mode; metrics_df reports both versions.
    """
    if u_mode not in ("rows", "incidence"):
        raise ValueError("u_mode must be 'rows' or 'incidence'")
    d = df_codes[["concept", "participant", "code"]]
    U_rows = d.groupby("concept").size()
    inc = d.drop_duplicates()
    U_inc = inc.groupby("concept").size()
    T_all = d.groupby("concept")["participant"].nunique()
    freq = inc.groupby(["concept", "code"]).size()      # participants per code
    Q1_all = (freq == 1).groupby(level=0).sum()
    Q2_all = (freq == 2).groupby(level=0).sum()

    losses, covs, rows = [], [], []
    for concept in U_rows.index:
        T, Q1, Q2 = int(T_all[concept]), int(Q1_all[concept]), int(Q2_all[concept])
        Ur, Ui = int(U_rows[concept]), int(U_inc[concept])
        C_rows = chao_coverage(Ur, T, Q1, Q2)
        C_inc = chao_coverage(Ui, T, Q1, Q2)
        C_hat = C_rows if u_mode == "rows" else C_inc
        covs.append(C_hat)
        losses.append((C_hat - C_target) ** 2)
        rows.append({"concept": concept,
                     "U": Ur if u_mode == "rows" else Ui,
                     "T": T, "Q1": Q1, "Q2": Q2,
                     "coverage": C_hat,
                     "U_rows": Ur, "U_incidence": Ui,
                     "coverage_rows": C_rows, "coverage_incidence": C_inc})
    metrics_df = pd.DataFrame(rows)
    return float(np.mean(losses)), float(np.mean(covs)), metrics_df

## 7) GRID SEARCH

Only in stage 1. The kNN is computed once with the largest `k` (identical results, much faster). A warning is shown if the optimum is on the edge of the grid.

In [ ]:

start = time.time()

grid_k          = [1, 5, 10, 15, 20, 25, 30, 35, 40]
grid_sim_min    = [0.33, 0.36, 0.39, 0.42, 0.45, 0.48, 0.51, 0.54, 0.57, 0.60]
grid_resolution = [0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0]

def grid_search(df_in, X, grid_k, grid_sim_min, grid_resolution,
                C_target=0.60, u_mode="rows", seed=42, legacy=False, verbose=False):
    """
    Same search as FACT 2.5 (loop k -> sim_min -> resolution, first minimum wins).
    legacy=True : sklearn kNN once per k (2.5 recomputed it for every k x sim_min;
                  results are identical, only faster).
    legacy=False: knn_search once with max(k) and the first k+1 columns for each k.
    """
    best = best_df_codes = best_metrics_df = None
    grid_rows = []
    if not legacy:
        nn_all = knn_search(X, max(grid_k))
    for k in grid_k:
        if legacy:
            nbrs = NearestNeighbors(n_neighbors=k+1, metric="cosine").fit(X)
            nn_k = nbrs.kneighbors(X)
        else:
            nn_k = (nn_all[0][:, :k+1], nn_all[1][:, :k+1])
        for s in grid_sim_min:
            G = knn_graph(X, k=k, sim_min=s, nn=nn_k)
            for r in grid_resolution:
                labels = louvain_labels(G, resolution=r, seed=seed)
                df_try = assign_singletons_as_unique(df_in, labels)
                L, Cbar, per_concept = cover_loss(df_try, C_target=C_target, u_mode=u_mode)
                grid_rows.append({"k": k, "sim_min": s, "resolution": r, "loss": L, "Cbar": Cbar,
                                  "Cbar_rows": per_concept["coverage_rows"].mean(),
                                  "Cbar_incidence": per_concept["coverage_incidence"].mean(),
                                  "n_codes": int(df_try["code"].nunique())})
                if verbose:
                    print(grid_rows[-1])
                if (best is None) or (L < best["loss"]):
                    best = {"loss": L, "Cbar": Cbar, "k": k, "sim_min": s, "resolution": r}
                    best_df_codes = df_try.copy()
                    best_metrics_df = per_concept.copy()
    return best, best_df_codes, best_metrics_df, pd.DataFrame(grid_rows)

def grid_edge_warnings(best, grid_k, grid_sim_min, grid_resolution):
    """Warn if the optimum sits on the border of the grid."""
    msgs = []
    for name, grid in (("k", grid_k), ("sim_min", grid_sim_min), ("resolution", grid_resolution)):
        if len(grid) > 1 and best[name] in (min(grid), max(grid)):
            side = "lower" if best[name] == min(grid) else "upper"
            msgs.append(f"Best {name}={best[name]} is on the {side} edge of the grid "
                        f"[{min(grid)}, {max(grid)}]; the true optimum may lie outside it.")
    for m in msgs:
        warnings.warn(m)
    return msgs

hyperparameter_search_time = 0.0
grid_df, edge_msgs = None, []
if STAGE == 1:
    if FIXED_PARAMS is None:
        best, best_df_codes, best_metrics_df, grid_df = grid_search(
            df, X, grid_k, grid_sim_min, grid_resolution,
            C_target=C_TARGET, u_mode=U_MODE, seed=SEED, legacy=LEGACY)
        edge_msgs = grid_edge_warnings(best, grid_k, grid_sim_min, grid_resolution)
    else:
        # fixed parameters, no grid (e.g. to reproduce published results)
        best, best_df_codes, best_metrics_df, grid_df = grid_search(
            df, X, [FIXED_PARAMS["k"]], [FIXED_PARAMS["sim_min"]], [FIXED_PARAMS["resolution"]],
            C_target=C_TARGET, u_mode=U_MODE, seed=SEED, legacy=LEGACY)

    # stage-1 bookkeeping: every row is a node of the state
    best_df_codes["stage"] = 1
    best_df_codes["row_id"] = np.arange(len(best_df_codes))
    best_df_codes["assign_method"] = "fit"
    best_df_codes["assign_sim"] = np.nan
    best_df_codes["is_new_code"] = True
    X_all = X

    print("BEST:",
          f"loss={best['loss']:.4f}",
          f"C̄={best['Cbar']:.3f}",
          f"k={best['k']} sim_min={best['sim_min']} resolution={best['resolution']}")
    end = time.time()
    hyperparameter_search_time = end - start
    print(f"Time of hyperparameter search: {hyperparameter_search_time}")
else:
    print("STAGE >= 2: grid search skipped; k, sim_min and resolution come from the stage-1 state (7b).")

## 7b) STAGE ≥ 2: LOAD STATE, VALIDATE, ASSIGN, RESIDUALS

1. Load the state; model and normalization must match (the model is also checked by re-embedding a few stored properties).
2. Concepts must be the same as in stage 1 (new concept → error; missing concept → warning).
3. Participant IDs must not collide with earlier stages; the input file must be new.
4. Each new property: its `k` nearest coded nodes with similarity ≥ `sim_min` vote with their similarity; largest sum wins (tie → most similar neighbor). → `knn_vote`
5. Properties without any neighbor ≥ `sim_min` (residuals): kNN graph + Louvain among themselves, same `k`, `sim_min`, `resolution` and seed. New codes start at max existing code + 1. → `new_cluster` / `singleton`
6. Coverage is recomputed on the accumulated data. Codes of earlier stages are never touched.

In [ ]:
start = time.time()

STATE_FILES = ("embeddings.npy", "nodes.csv", "params.json", "data_hashes.json", "code_medoids.csv")
NODE_COLS = ["row_id", "concept", "participant", "stage", "property", "property_norm",
             "code", "assign_method", "assign_sim", "participant_raw"]

def load_state(state_dir):
    missing = [f for f in STATE_FILES if not os.path.exists(os.path.join(state_dir, f))]
    if missing:
        raise FileNotFoundError(f"State in {state_dir} is incomplete, missing: {missing}. "
                                "Run the notebook with STAGE = 1 first.")
    nodes = pd.read_csv(os.path.join(state_dir, "nodes.csv"),
                        dtype={"participant": str, "participant_raw": str},
                        keep_default_na=False, na_values={"assign_sim": [""]})
    X = np.load(os.path.join(state_dir, "embeddings.npy"))
    with open(os.path.join(state_dir, "params.json")) as f:
        params = json.load(f)
    with open(os.path.join(state_dir, "data_hashes.json")) as f:
        hashes = json.load(f)
    medoids = pd.read_csv(os.path.join(state_dir, "code_medoids.csv"), keep_default_na=False)
    if len(nodes) != len(X):
        raise ValueError(f"State is corrupt: {len(nodes)} nodes but {len(X)} embeddings.")
    if not (nodes["row_id"].values == np.arange(len(nodes))).all():
        raise ValueError("State is corrupt: row_id is not 0..n-1 in file order.")
    return {"nodes": nodes, "X": X, "params": params, "hashes": hashes, "medoids": medoids}

def rollback_state(state, stage):
    """Keep only stages < stage (lets you re-run a stage). Frozen codes are untouched."""
    nodes = state["nodes"]
    last = int(nodes["stage"].max())
    if last < stage - 1:
        raise ValueError(f"State only has stages up to {last}; run STAGE = {last + 1} first.")
    if last >= stage:
        warnings.warn(f"State already contains stage(s) {stage}..{last}; they are dropped "
                      f"and stage {stage} is coded again from stage {stage - 1}.")
        keep = (nodes["stage"] < stage).values
        state = dict(state)
        state["nodes"] = nodes[keep].reset_index(drop=True)
        state["X"] = state["X"][keep]
        med = state["medoids"]
        state["medoids"] = med[med["stage_created"] < stage].reset_index(drop=True)
        state["hashes"] = {s: h for s, h in state["hashes"].items() if int(s) < stage}
        sizes = state["nodes"].groupby("code").size()
        state["medoids"]["n_nodes"] = state["medoids"]["code"].map(sizes).astype(int)
    return state

def validate_state_params(params, model_name, norm_config, legacy, u_mode):
    if params["model"] != model_name:
        raise ValueError(f"Model mismatch: state used {params['model']!r}, now {model_name!r}.")
    if params["normalization"] != norm_config:
        diff = {k: (params["normalization"].get(k), v) for k, v in norm_config.items()
                if params["normalization"].get(k) != v}
        raise ValueError(f"Normalization mismatch (saved, now): {diff}")
    if params.get("legacy") != legacy:
        warnings.warn(f"LEGACY was {params.get('legacy')} in stage 1 and is {legacy} now.")
    if params["u_mode"] != u_mode:
        warnings.warn(f"u_mode was {params['u_mode']!r} in stage 1; using it again (not {u_mode!r}).")

def validate_embeddings_probe(state, encoder=None, n_probe=32, min_cos=0.999):
    """Re-embed a few stored properties; the model must give the same vectors."""
    nodes, X = state["nodes"], state["X"]
    pick = np.unique(np.linspace(0, len(nodes) - 1, min(n_probe, len(nodes))).astype(int))
    Z = encode_passages_en(nodes["property_norm"].iloc[pick].astype(str).tolist(), encoder=encoder)
    cos = np.sum(Z * X[pick], axis=1) / (np.linalg.norm(Z, axis=1) * np.linalg.norm(X[pick], axis=1))
    if cos.min() < min_cos:
        raise ValueError(f"Embeddings differ from the stored ones (min cosine {cos.min():.5f}). "
                         "Same model name but different weights/version?")
    return float(cos.min())

def validate_concepts(new_concepts, ref_concepts):
    new_c, ref_c = set(new_concepts), set(ref_concepts)
    extra, missing = sorted(new_c - ref_c), sorted(ref_c - new_c)
    if extra:
        raise ValueError(f"{len(extra)} concept(s) not present in stage 1: {extra}")
    if missing:
        warnings.warn(f"{len(missing)} stage-1 concept(s) have no data in this stage: {missing}")
    return extra, missing

def check_new_data(df_new, state, sha):
    clash = sorted(set(df_new["participant"]) & set(state["nodes"]["participant"]))
    if clash:
        raise ValueError(f"Participant IDs collide with earlier stages: {clash[:10]}")
    for s, h in state["hashes"].items():
        if h["sha256"] == sha:
            raise ValueError(f"This file is identical to the stage {s} input ({h['file']}).")

def assign_to_existing_codes(X_new, X_old, codes_old, k, sim_min):
    """
    Each new property looks at its k nearest coded nodes; neighbors with sim >= sim_min vote
    with their similarity. Winner = largest sum; tie -> code of the most similar neighbor.
    Returns codes (-1 = residual) and assign_sim (similarity to the closest neighbor of
    the winning code).
    """
    codes_old = np.asarray(codes_old)
    out_code = np.full(len(X_new), -1, dtype=int)
    out_sim = np.full(len(X_new), np.nan)
    if len(X_new) == 0:
        return out_code, out_sim
    dist, idx = knn_search(X_old, k, Y=X_new)
    sim = 1.0 - dist
    for i in range(len(X_new)):
        ok = sim[i] >= sim_min
        if not ok.any():
            continue
        nb_codes, nb_sims = codes_old[idx[i][ok]], sim[i][ok]    # sorted by sim, desc
        totals = {}
        for c, s in zip(nb_codes, nb_sims):
            totals[c] = totals.get(c, 0.0) + s
        top = max(totals.values())
        tied = {c for c, t in totals.items() if t >= top - 1e-9}
        for c, s in zip(nb_codes, nb_sims):
            if c in tied:
                out_code[i], out_sim[i] = int(c), float(s)
                break
    return out_code, out_sim

def code_residuals(X_res, k, sim_min, resolution, seed, start_code):
    """kNN graph + Louvain among residuals only; new codes start at start_code."""
    n = len(X_res)
    if n == 0:
        return np.array([], dtype=int), np.array([], dtype=object)
    if n == 1:
        labels = np.array([0])
    else:
        G = knn_graph(X_res, k=k, sim_min=sim_min, nn=knn_search(X_res, k))
        labels = louvain_labels(G, resolution=resolution, seed=seed)
    uniq, first = np.unique(labels, return_index=True)
    order = uniq[np.argsort(first)]                     # number codes by first appearance
    remap = {lab: start_code + r for r, lab in enumerate(order)}
    codes = np.array([remap[l] for l in labels], dtype=int)
    sizes = pd.Series(labels).value_counts()
    methods = np.where(sizes.loc[labels].values >= 2, "new_cluster", "singleton").astype(object)
    return codes, methods

def code_new_stage(df_new, X_new, nodes_old, X_old, k, sim_min, resolution, seed, stage):
    """Stage >= 2: assign to frozen codes, then cluster the residuals."""
    out = df_new.copy()
    codes, sims = assign_to_existing_codes(X_new, X_old, nodes_old["code"].values, k, sim_min)
    method = np.where(codes >= 0, "knn_vote", "").astype(object)
    res = np.flatnonzero(codes < 0)
    start = int(nodes_old["code"].max()) + 1
    res_codes, res_methods = code_residuals(X_new[res], k, sim_min, resolution, seed, start)
    codes[res], method[res] = res_codes, res_methods
    out["code"], out["assign_method"], out["assign_sim"] = codes, method, sims
    out["stage"] = stage
    out["is_new_code"] = out["code"] >= start
    return out

def compute_medoids(nodes, X, codes=None, stage_created=None):
    """Representative property of each code: member closest to the code centroid."""
    rows = []
    groups = nodes.groupby("code").indices
    for c in (sorted(groups) if codes is None else codes):
        m = groups[c]
        centroid = X[m].sum(axis=0)
        best_i = m[int(np.argmax(X[m] @ centroid))]
        rows.append({"code": int(c),
                     "stage_created": int(nodes["stage"].iloc[m].min()) if stage_created is None else stage_created,
                     "medoid_row_id": int(nodes["row_id"].iloc[best_i]),
                     "medoid_property": nodes["property"].iloc[best_i],
                     "medoid_property_norm": nodes["property_norm"].iloc[best_i],
                     "n_nodes": len(m)})
    return pd.DataFrame(rows, columns=["code", "stage_created", "medoid_row_id", "medoid_property",
                                       "medoid_property_norm", "n_nodes"])

def update_medoids(medoids_old, nodes_all, X_all, stage):
    """Old codes keep their medoid (stable labels); only new codes get one."""
    med = medoids_old.copy()
    sizes = nodes_all.groupby("code").size()
    med["n_nodes"] = med["code"].map(sizes).astype(int)
    new_codes = sorted(set(nodes_all["code"]) - set(med["code"]))
    new_med = compute_medoids(nodes_all, X_all, codes=new_codes, stage_created=stage)
    return pd.concat([med, new_med], ignore_index=True)

def metrics_by_stage(nodes_all, C_target=0.60, u_mode="rows"):
    """One row per concept and stage, on cumulative data (stages 1..s), with change vs s-1."""
    out = []
    for s in sorted(nodes_all["stage"].unique()):
        cum = nodes_all[nodes_all["stage"] <= s]
        _, _, m = cover_loss(cum, C_target=C_target, u_mode=u_mode)
        m = m.merge(cum.groupby("concept")["code"].nunique().reset_index(name="n_codes"),
                    on="concept", how="left")
        m.insert(1, "stage", int(s))
        out.append(m)
    out = pd.concat(out, ignore_index=True).sort_values(["concept", "stage"]).reset_index(drop=True)
    out["coverage_change"] = out.groupby("concept")["coverage"].diff()
    return out

def save_state(state_dir, nodes, X, params, hashes, medoids):
    os.makedirs(state_dir, exist_ok=True)
    np.save(os.path.join(state_dir, "embeddings.npy"), np.asarray(X, dtype=np.float32))
    nodes[NODE_COLS].to_csv(os.path.join(state_dir, "nodes.csv"), index=False)
    with open(os.path.join(state_dir, "params.json"), "w") as f:
        json.dump(params, f, indent=2)
    with open(os.path.join(state_dir, "data_hashes.json"), "w") as f:
        json.dump(hashes, f, indent=2)
    medoids.to_csv(os.path.join(state_dir, "code_medoids.csv"), index=False)

assignment_time = 0.0
if STAGE >= 2:
    # 1) state + model/normalization checks
    state = load_state(STATE_DIR)
    state = rollback_state(state, STAGE)          # only matters if this stage is run twice
    params = state["params"]
    validate_state_params(params, MODEL_NAME, NORM_CONFIG, LEGACY, U_MODE)
    U_MODE, C_TARGET, SEED = params["u_mode"], params["C_target"], params["seed"]
    probe_cos = validate_embeddings_probe(state)
    nodes_prev, X_prev = state["nodes"].copy(), state["X"].copy()

    # 2) same concepts as stage 1
    validate_concepts(df["concept"].unique(),
                      nodes_prev.loc[nodes_prev["stage"] == 1, "concept"].unique())
    # 3) participants and file
    check_new_data(df, state, input_sha256)

    # 4) + 5) assign to frozen codes, then residuals
    k, s, r = params["k"], params["sim_min"], params["resolution"]
    df_new = df.copy()
    df_new["row_id"] = np.arange(len(df_new)) + len(nodes_prev)
    df_new = code_new_stage(df_new, X, nodes_prev, X_prev, k, s, r, SEED, STAGE)

    # 6) accumulated data (earlier stages exactly as saved)
    prev_codes = nodes_prev.copy()
    prev_codes["is_new_code"] = False
    best_df_codes = pd.concat([prev_codes, df_new], ignore_index=True)
    created = dict(zip(state["medoids"]["code"], state["medoids"]["stage_created"]))
    best_df_codes["is_new_code"] = [created.get(c, STAGE) == st
                                    for c, st in zip(best_df_codes["code"], best_df_codes["stage"])]
    X_all = np.vstack([X_prev, X])

    L, Cbar, best_metrics_df = cover_loss(best_df_codes, C_target=C_TARGET, u_mode=U_MODE)
    best = {"loss": L, "Cbar": Cbar, "k": k, "sim_min": s, "resolution": r}

    end = time.time()
    assignment_time = end - start
    n_new_codes = int(df_new.loc[df_new["is_new_code"], "code"].nunique())
    print(f"Stage {STAGE}: {len(df_new)} new properties | k={k} sim_min={s} resolution={r}")
    print(df_new["assign_method"].value_counts().to_string())
    print(f"New codes: {n_new_codes} | accumulated C̄={Cbar:.3f} (loss={L:.4f}) | "
          f"embedding probe min cos={probe_cos:.5f}")
    print(f"Time of assignment: {assignment_time}")
else:
    print("STAGE == 1: nothing to assign.")

## 8) SAVE OUTPUT FILES

In [ ]:

CODE_COLS = ["concept", "participant", "property", "property_norm", "code",
             "participant_raw", "stage", "row_id", "assign_method", "assign_sim", "is_new_code"]

# (a) full coded table (all stages)
codes_path = f"{PREFIX}_codes.csv"
best_df_codes[CODE_COLS].to_csv(codes_path, index=False)

# (b) per-concept metrics, one row per concept and stage (accumulated data), with coverage change
metrics_df = metrics_by_stage(best_df_codes, C_target=C_TARGET, u_mode=U_MODE)
metrics_path = f"{PREFIX}_metrics_per_concept.csv"
metrics_df.to_csv(metrics_path, index=False)

# (c) per (concept, code) sizes table (helpful for QC)
sizes = (best_df_codes.groupby(["concept","code"])["property"]
         .count().reset_index(name="n_properties")
         .sort_values(["concept","n_properties"], ascending=[True, False]))
sizes_path = f"{PREFIX}_code_sizes.csv"
sizes.to_csv(sizes_path, index=False)

# (d) state for the next stage
if STAGE == 1:
    if os.path.exists(os.path.join(STATE_DIR, "nodes.csv")):
        warnings.warn(f"Overwriting the existing state in {STATE_DIR} (STAGE = 1).")
    params = {"fact_version": FACT_VERSION, "k": int(best["k"]), "sim_min": float(best["sim_min"]),
              "resolution": float(best["resolution"]), "seed": SEED, "model": MODEL_NAME,
              "normalization": NORM_CONFIG, "C_target": C_TARGET, "u_mode": U_MODE, "legacy": LEGACY,
              "data_name": DATA_NAME,
              "grid": ({"k": grid_k, "sim_min": grid_sim_min, "resolution": grid_resolution}
                       if FIXED_PARAMS is None else {"fixed": FIXED_PARAMS}),
              "created": datetime.datetime.now().isoformat(timespec="seconds")}
    hashes = {}
    medoids = compute_medoids(best_df_codes, X_all)
else:
    hashes = dict(state["hashes"])
    medoids = update_medoids(state["medoids"], best_df_codes, X_all, STAGE)
hashes[str(STAGE)] = {"file": CSV_PATH, "sha256": input_sha256, "n_rows": int(len(df))}
save_state(STATE_DIR, best_df_codes, X_all, params, hashes, medoids)

# (e) summary JSON
summary = {
    "best": best,
    "stage": STAGE,
    "n_rows": int(len(best_df_codes)),
    "n_rows_by_stage": {s: h["n_rows"] for s, h in hashes.items()},
    "n_codes_total": int(best_df_codes["code"].nunique()),
    "n_new_codes_this_stage": int(best_df_codes.loc[(best_df_codes["stage"] == STAGE) & best_df_codes["is_new_code"], "code"].nunique()),
    "assign_method_this_stage": best_df_codes.loc[best_df_codes["stage"] == STAGE, "assign_method"].value_counts().to_dict(),
    "u_mode": U_MODE,
    "Cbar_rows": float(best_metrics_df["coverage_rows"].mean()),
    "Cbar_incidence": float(best_metrics_df["coverage_incidence"].mean()),
    "legacy": LEGACY,
    "grid_edge_warnings": edge_msgs,
    "fact_version": FACT_VERSION,
}
with open(f"{PREFIX}_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# (f) time JSON
total_time = {
    "standardization_time": standardization_time,
    "embedding_time": embedding_time,
    "hyperparameter_search_time": hyperparameter_search_time,
    "assignment_time": assignment_time,
    "total": standardization_time+embedding_time+hyperparameter_search_time+assignment_time
}
with open(f"{PREFIX}_time.json", "w") as f:
    json.dump(total_time, f, indent=2)

# (g) full grid (stage 1)
if grid_df is not None:
    grid_df.to_csv(f"{PREFIX}_grid.csv", index=False)

print("✅ Files saved:")
for p in [codes_path, metrics_path, sizes_path, f"{PREFIX}_summary.json", f"{PREFIX}_time.json"]:
    print(" -", p)
if grid_df is not None:
    print(" -", f"{PREFIX}_grid.csv")
print(" - state:", STATE_DIR, "->", ", ".join(STATE_FILES))

## 9) GRAPHS (inline + saved to disk)


In [ ]:

# 9a) overall histogram of cluster sizes (all concepts together)
overall_sizes = best_df_codes["code"].value_counts().sort_values(ascending=False)
plt.figure(figsize=(8,4))
plt.hist(overall_sizes.values, bins=30)
plt.title(f"Distribution of cluster sizes (all concepts, stages 1-{STAGE})")
plt.xlabel("cluster size (#properties)")
plt.ylabel("#clusters")
plt.tight_layout()
plt.savefig(f"{PREFIX}_overall_cluster_size_hist.png", dpi=150)
plt.show()

# 9b) per-concept bar charts of top codes (into a single multipage PDF); new codes of this stage in orange
new_codes_now = set(best_df_codes.loc[(best_df_codes["stage"] == STAGE) & best_df_codes["is_new_code"], "code"]) if STAGE >= 2 else set()
pdf_path = f"{PREFIX}_per_concept_top_codes.pdf"
with PdfPages(pdf_path) as pdf:
    for concept, grp in sizes.groupby("concept"):
        top = grp.nlargest(20, "n_properties")  # top 20 codes per concept
        plt.figure(figsize=(8, max(3, 0.35*len(top))))
        plt.barh(top["code"].astype(str), top["n_properties"],
                 color=["tab:orange" if c in new_codes_now else "tab:blue" for c in top["code"]])
        plt.gca().invert_yaxis()
        plt.title(f"{concept} — top codes by #properties")
        plt.xlabel("#properties")
        plt.ylabel("code")
        plt.tight_layout()
        pdf.savefig()
        plt.close()

# 9c) stage >= 2: coverage per concept, previous vs current stage
if STAGE >= 2:
    wide = metrics_df.pivot(index="concept", columns="stage", values="coverage")
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
    ax[0].scatter(wide[STAGE - 1], wide[STAGE], s=12)
    lo, hi = np.nanmin(wide.values), np.nanmax(wide.values)
    ax[0].plot([lo, hi], [lo, hi], color="grey", lw=1)
    ax[0].set_xlabel(f"coverage, stage {STAGE - 1}")
    ax[0].set_ylabel(f"coverage, stages 1-{STAGE}")
    ax[0].set_title("Coverage per concept")
    cnt = df_new["assign_method"].value_counts()
    ax[1].bar(cnt.index, cnt.values)
    ax[1].set_title(f"Stage {STAGE}: how new properties were coded")
    ax[1].set_ylabel("#properties")
    plt.tight_layout()
    plt.savefig(f"{PREFIX}_coverage_by_stage.png", dpi=150)
    plt.show()

print("✅ Plots saved:")
print(" -", f"{PREFIX}_overall_cluster_size_hist.png")
print(" -", pdf_path)
if STAGE >= 2:
    print(" -", f"{PREFIX}_coverage_by_stage.png")

# 9d) show a quick preview table inline (like “all rows are coded”)
display(best_df_codes.sort_values(["concept","code","property"]).head(20))

## 10) CHECKS

Each `assert` stops the notebook if something is wrong.

In [ ]:

ALLOWED_METHODS = {"fit", "knn_vote", "new_cluster", "singleton"}

# (1) rows of codes.csv == sum of rows of all stages
n_by_stage = {int(s): h["n_rows"] for s, h in hashes.items()}
codes_check = pd.read_csv(codes_path, dtype={"participant": str}, keep_default_na=False)
assert len(codes_check) == sum(n_by_stage.values()), "codes.csv rows != sum of stage rows"
assert best_df_codes.groupby("stage").size().to_dict() == n_by_stage

# (2) no concepts outside stage 1
stage1_concepts = set(best_df_codes.loc[best_df_codes["stage"] == 1, "concept"])
assert set(best_df_codes["concept"]) <= stage1_concepts, "new concepts appeared after stage 1"

# (3) participant IDs of different stages do not collide
assert best_df_codes.groupby("participant")["stage"].nunique().max() == 1, "participant in >1 stage"

# (4) every row coded, valid methods
assert (best_df_codes["code"] >= 0).all()
assert set(best_df_codes["assign_method"]) <= ALLOWED_METHODS

# (5) stage >= 2: earlier codes identical to the saved state (in memory and on disk)
if STAGE >= 2:
    prev = best_df_codes[best_df_codes["stage"] < STAGE]
    assert (prev["row_id"].values == nodes_prev["row_id"].values).all()
    assert (prev["code"].values == nodes_prev["code"].values).all(), "a frozen code changed!"
    st = load_state(STATE_DIR)
    old = st["nodes"]["stage"].values < STAGE
    assert (st["nodes"].loc[old, "code"].values == nodes_prev["code"].values).all()
    assert np.array_equal(st["X"][old], X_prev), "stored embeddings of earlier stages changed"
    assert (best_df_codes.loc[best_df_codes["is_new_code"] & (best_df_codes["stage"] == STAGE), "code"]
            > nodes_prev["code"].max()).all()
    # coverage of earlier stages is unchanged by this stage
    m_prev = metrics_by_stage(nodes_prev, C_target=C_TARGET, u_mode=U_MODE)
    m_now = metrics_df[metrics_df["stage"] < STAGE].reset_index(drop=True)
    assert np.allclose(m_prev["coverage"].values, m_now["coverage"].values)

# (6) stage 1: re-coding from scratch with the best parameters gives the same codes
#     (checks that the fast grid = building the kNN graph for that k alone)
if STAGE == 1:
    nn_best = None if LEGACY else knn_search(X, best["k"])
    G_chk = knn_graph(X, k=best["k"], sim_min=best["sim_min"], nn=nn_best)
    lab_chk = louvain_labels(G_chk, resolution=best["resolution"], seed=SEED)
    assert (assign_singletons_as_unique(df, lab_chk)["code"].values == best_df_codes["code"].values).all()

# (7) LEGACY: same rows/normalization as FACT 2.5 and same C̄ as the published results
REF_CODES = f"../data/results/en/en_{DATA_NAME}_codes.csv"
REF_SUMMARY = f"../data/results/en/en_{DATA_NAME}_summary.json"
if LEGACY and STAGE == 1 and os.path.exists(REF_CODES) and os.path.exists(REF_SUMMARY):
    ref = pd.read_csv(REF_CODES)
    with open(REF_SUMMARY) as f:
        ref_best = json.load(f)["best"]
    assert len(ref) == len(df) and (ref["concept"].values == df["concept"].values).all()
    assert (ref["participant"].astype(str).values == df["participant_raw"].astype(str).values).all()
    assert (ref["property_norm"].fillna("").values == df["property_norm"].values).all()
    assert abs(cover_loss(ref, C_target=C_TARGET, u_mode="rows")[1] - ref_best["Cbar"]) < 1e-12
    print(f"LEGACY: rows, normalization and cover_loss match FACT 2.5 (published C̄={ref_best['Cbar']:.4f})")
    if FIXED_PARAMS == {p: ref_best[p] for p in ("k", "sim_min", "resolution")}:
        ari = adjusted_rand_score(ref["code"], best_df_codes["code"])
        print(f"LEGACY fixed params: C̄={best['Cbar']:.4f} vs published {ref_best['Cbar']:.4f}; "
              f"ARI vs published codes = {ari:.4f}")
        assert abs(best["Cbar"] - ref_best["Cbar"]) < 0.005, "C̄ does not reproduce FACT 2.5"

print(f"✅ All checks passed (stage {STAGE}).")